In [ ]:
import re, string, random, hashlib
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag, bigrams

for p in ["punkt", "punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(p, quiet=True)

stop_words = set(stopwords.words("english"))

if "generate_dataset" not in globals():
    pool = [
        "My laptop keeps crashing after the 2.0.1 update. Please email me at john.doe@mail.com or call 555-123-4567.",
        "The Wi-Fi isn't working in room 12. Signal strength is 95.5 percent but pages keep freezing.",
        "Printer is offline again. I've restarted it twice, but the error is still there. See https://help.example.com/printer for logs.",
        "Our VPN client version 3.1.4 disconnects every hour. Users in Delhi are affected.",
        "Outlook crashes when I open attachments. Contact me on +91 98765 43210 for a quick call.",
        "The state-of-the-art monitor flickers during video calls. Brightness was set to 80.5 percent.",
        "I can't login to the portal. The password reset email never arrived at amit_k@college.edu.in.",
        "Installed the new driver 5.2.0 yesterday and the connection became worse. Earlier it was better.",
        "Keyboard keys are sticking and some letters aren't typing. Replacement requests were ignored by the helpdesk.",
        "The server room temperature reached 31.5 degrees. Cooling units are running but alarms are still ringing.",
        "Antivirus updates failed with error 0x80070005. Visit http://support.example.org/av/help for the fix.",
        "Zoom meetings are lagging badly and audio keeps dropping. Network engineers have been informed.",
        "Shared drive is not mapping on Windows 11. Call the desk at 044-555-0199 if urgent.",
        "My account got locked after three attempts. Unlocking takes hours, which isn't acceptable.",
        "The projector in the seminar hall displays distorted colors. Version 1.9.8 of the firmware is installed.",
        "Backup jobs are running slower than before. The disk is at 92.5 percent capacity and filling quickly.",
        "Software licenses expired yesterday and users are getting warnings. Please renew them soon.",
        "Webcam isn't detected after the 22H2 update. Mail it to it-help@company.com when fixed.",
        "Browser keeps freezing while uploading files bigger than 25.5 MB. Chrome 118.0.2 is being used.",
        "Database queries are timing out during peak hours. The monitoring dashboard shows high memory usage.",
    ]

    def generate_dataset(roll):
        r = random.Random(roll)
        return r.sample(pool, 14)

    def process_ticket(text):
        sents = sent_tokenize(text)
        toks = word_tokenize(text)
        punct = [t for t in toks if all(ch in string.punctuation for ch in t)]
        stops = [t for t in toks if t.lower() in stop_words]
        final = [t.lower() for t in toks if t not in punct and t.lower() not in stop_words and not t.startswith("'") and t.lower() != "n't"]
        return {"sentences": sents, "tokens": toks, "punct": punct, "stopwords": stops, "final": final}

    TEST_LINES = [
        "Contact john.doe@mail.com or call 555-123-4567.",
        "Visit https://example.com/help?id=7 for the state-of-the-art guide.",
        "The Wi-Fi isn't working and scored 95.5 on version 2.0.1.",
        "An elephant ate Orange pie at 10.25 near www.test.org today.",
        "Call +91 98765 43210 or write to a_b@x.co.in",
    ]

roll = 127
tickets = generate_dataset(roll)
print(hashlib.md5(str(tickets).encode()).hexdigest()[:8])
print(len(tickets))

In [ ]:
rows = []
for i, t in enumerate(tickets):
    p = process_ticket(t)
    rows.append({
        "ticket": i + 1,
        "sentences": len(p["sentences"]),
        "tokens": len(p["tokens"]),
        "punctuation": len(p["punct"]),
        "stopwords": len(p["stopwords"]),
        "unique_tokens": len(set(p["tokens"])),
        "final_tokens": len(p["final"]),
    })
table = pd.DataFrame(rows)
total = table.drop(columns="ticket").sum()
total["ticket"] = "total"
table = pd.concat([table, total.to_frame().T], ignore_index=True)
print(table.to_string(index=False))

p = process_ticket(tickets[0])
print(tickets[0])
print(p["tokens"])
print(p["punct"])
print(p["stopwords"])
print(p["final"])

In [ ]:
alltext = " ".join(tickets)
pat_long = r"\b[A-Za-z]{6,}\b"
pat_num = r"\b\d+(?:\.\d+)*\b"
pat_cap = r"\b[A-Z][a-z]+\b"
pat_vowel = r"\b[aeiouAEIOU][A-Za-z]*\b"
pat_email = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
pat_url = r"(?:https?://|www\.)\S+?(?=[.,;!?]*(?:\s|$))"
pat_phone = r"\+\d{1,3}[-.\s]\d{4,5}[-.\s]\d{5}|\b\d{3,4}[-.]\d{3}[-.]\d{4}\b"

for name, text in (("tickets", alltext), ("test_lines", " ".join(TEST_LINES))):
    print(name)
    print(re.findall(pat_long, text))
    print(re.findall(pat_num, text))
    print(re.findall(pat_cap, text))
    print(re.findall(pat_vowel, text))

def clean(text):
    text = re.sub(pat_email, "<EMAIL>", text)
    text = re.sub(pat_url, "<URL>", text)
    text = re.sub(pat_phone, "<PHONE>", text)
    return text

for line in TEST_LINES:
    print(clean(line))
for t in tickets:
    print(clean(t))

In [ ]:
pattern = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+|(?:https?://|www\.)[^\s]*[^\s.,;!?]|\+\d{1,3}[-.\s]\d{4,5}[-.\s]\d{5}|\b\d{3,4}[-.]\d{3}[-.]\d{4}\b|\d+(?:\.\d+)+|\w+(?:[-']\w+)*|[^\w\s]"
tk = RegexpTokenizer(pattern)

def my_tokenize(text):
    return tk.tokenize(text)

for line in TEST_LINES + tickets:
    mine = my_tokenize(line)
    nl = word_tokenize(line)
    print(mine)
    print(nl)
    print([t for t in mine if t not in nl], [t for t in nl if t not in mine])
    print()

print("1. contractions: word_tokenize splits isn't into is and n't, my tokenizer keeps isn't as one token")
print("2. urls and emails: word_tokenize splits an email at the @ sign into three tokens and a url into pieces like https, :, //example.com/help, ? and id=7, my tokenizer keeps each as one token")
print("3. phone numbers written with spaces: word_tokenize splits +91 98765 43210 into three tokens, my tokenizer keeps the whole phone number as one token")

In [ ]:
finals = []
for t in tickets:
    finals += process_ticket(t)["final"]
words = sorted(set(finals))

porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp = RegexpStemmer("ing$", min=4)
lemmatizer = WordNetLemmatizer()

def wn_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

tagged = pos_tag(words)
rows = []
for w, tag in tagged:
    rows.append({
        "word": w,
        "porter": porter.stem(w),
        "lancaster": lancaster.stem(w),
        "regexp": regexp.stem(w),
        "lemma": lemmatizer.lemmatize(w, wn_pos(tag)),
    })
stem_table = pd.DataFrame(rows)
diff = stem_table[stem_table[["porter", "lancaster", "regexp", "lemma"]].nunique(axis=1) > 1]
print(diff.to_string(index=False))
print([w for w in words if w.endswith("ing")])
print("suffix ing was chosen because crashing and freezing appear in the tickets, and removing ing gives crash and freez, which are close to the base verbs")

In [ ]:
s1 = []
s2 = []
s3 = []
for t in tickets:
    p = process_ticket(t)
    s1 += p["tokens"]
    s2 += [x for x in p["tokens"] if x not in p["punct"]]
    s3 += p["final"]
s4 = [porter.stem(x) for x in s3]

stats = []
for name, s in (("s1", s1), ("s2", s2), ("s3", s3), ("s4", s4)):
    stats.append({"set": name, "total": len(s), "unique": len(set(s)), "ttr": len(set(s)) / len(s)})
stats = pd.DataFrame(stats)
print(stats.to_string(index=False))

print(Counter(s1).most_common(10))
print(Counter(s3).most_common(10))

freq = sorted(Counter(s3).values(), reverse=True)
ranks = list(range(1, len(freq) + 1))
plt.plot(ranks, freq, marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("log rank")
plt.ylabel("log frequency")
plt.show()

print(Counter(bigrams(s1)).most_common(5))
print(Counter(bigrams(s3)).most_common(5))

In [ ]:
print("a)", stats["ttr"].round(3).tolist())
print("ttr changes because s2 removes repeated punctuation, s3 also removes frequent stopwords and lowercases, and s4 merges word forms through stemming, so the token counts and unique counts both shrink at different rates")

print("b)", word_tokenize("the wi-fi isn't working"), my_tokenize("the wi-fi isn't working"))
print("word_tokenize splits isn't into is and n't while my tokenizer keeps it as one contraction token")

bad = [(w, porter.stem(w)) for w in words if wordnet.synsets(porter.stem(w)) == [] and porter.stem(w) != w]
print("c)", bad[:5])
print("the stemmer cuts words to stems that are not real words, so the meaning is hard to read")

good = stem_table[(stem_table["lemma"] != stem_table["porter"]) & (stem_table["lemma"] != stem_table["word"])]
print("d)", good[["word", "porter", "lemma"]].head(5).to_string(index=False))
print("the pos-aware lemmatizer returns a real dictionary word with the correct meaning, while the stemmer only chops the ending")